# Lesson 01 - Uvod v AI agente

Dobrodošli na prvem pouku v tečaju **AI agenti za začetnike**!

**AI agent** je program, ki uporablja velik jezikovni model (LLM) kot svoj mehanizem sklepanja in lahko izvaja *akcije* v resničnem svetu — kliče API-je, poizveduje baze podatkov ali izvaja kodo — da doseže cilj v imenu uporabnika.

V tem zvezku boste ustvarili svojega prvega agenta: **Potovalni agent**, ki priporoča počitniške destinacije. Medtem boste izvedeli, kako:

1. Povezati se z Azure AI Foundry Agent Service z uporabo **Microsoft Agent Frameworka**.
2. Agentu dati **orodje** — preprosto Python funkcijo, ki jo lahko kliče.
3. Zaženete agenta in pregledate njegov odgovor.
4. Pretakate agentov odgovor token za tokenom.


## Namestitev

Pred zagonom tega zapisa se prepričajte, da imate:

1. **Projekt Azure AI Foundry** z nameščenim klepetalnim modelom (npr. `gpt-4o-mini`).
2. **Prijavljeni v Azure CLI** — v terminalu zaženite `az login`.
3. **Nastavljene zahtevane okoljske spremenljivke:**
   - `AZURE_AI_PROJECT_ENDPOINT` — konec vašega projekta Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.

Spodnja celica namesti potrebne Python pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ustvarjanje prvega agenta

Agent potrebuje dve stvari:

- **Navodila**, ki mu povedo *kdo je* in *kako naj se obnaša* (sistemski poziv).
- **Orodja** — Python funkcije okrašene z `@tool`, ki jih agent lahko pokliče za pridobivanje informacij ali izvajanje dejanj.

Spodaj definiramo preprosto orodje, ki vrne seznam priljubljenih počitniških destinacij. Agent bo uporabil to orodje, ko uporabnik zaprosi za priporočila za potovanje.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Tokovni odzivi

Za bolj interaktivno izkušnjo lahko **tokovno** prikažete odziv agenta. Namesto da bi čakali na celoten odgovor, agent sproti daje dele besedila, ko so ti ustvarjeni. To je še posebej uporabno v klepetalnih vmesnikih, kjer želite izhod prikazovati v realnem času.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili:

- **Ustvariti ponudnika**, ki se poveže z Azure AI Foundry Agent Service prek `AzureAIProjectAgentProvider`.
- **Določiti orodje** z uporabo dekoratorja `@tool`, da lahko agent kliče vaše Python funkcije.
- **Zagnati agenta** z uporabniškim sporočilom in izpisati njegov odziv.
- **Tokovno predvajati odzive** za izhod v realnem času.

V naslednji lekciji bomo poglobljeno raziskali agentske okvire in se naučili, kako agentom dati močnejša orodja in zmožnosti večstopenjskega sklepanja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da imate v mislih, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v izvirnem jeziku velja za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za kakršne koli nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
